# Chapter 3 Practical 03: User-User kNN Prediction

Learning objectives:
- Select nearest neighbors for a target user.
- Predict missing ratings with a mean-centered, similarity-weighted formula.
- Explain a prediction through neighbor contributions.
- Generate Top-N user-user CF recommendations.

Slide connection: user-user CF, making predictions, prediction formula, and explainable memory-based CF.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("data")
if not (DATA_DIR / "ratings_chapter3.csv").exists():
    DATA_DIR = Path("../data")
if not (DATA_DIR / "ratings_chapter3.csv").exists():
    DATA_DIR = Path("chapter_03_collaborative_filtering/data")

ratings = pd.read_csv(DATA_DIR / "ratings_chapter3.csv")
movies = pd.read_csv(DATA_DIR / "movies_chapter3.csv")
ratings_named = ratings.merge(movies, on="movie_id", how="left")
rating_matrix = ratings_named.pivot_table(index="user_id", columns="title", values="rating")
rating_matrix


In [ ]:
def pearson_on_overlap(matrix, user_a, user_b):
    pair = matrix.loc[[user_a, user_b]].dropna(axis=1)
    if pair.shape[1] < 2:
        return np.nan
    if pair.loc[user_a].std() == 0 or pair.loc[user_b].std() == 0:
        return np.nan
    return float(np.corrcoef(pair.loc[user_a], pair.loc[user_b])[0, 1])

def user_neighbors(matrix, target_user, min_overlap=2):
    rows = []
    for other in matrix.index.drop(target_user):
        overlap = matrix.loc[[target_user, other]].notna().all(axis=0).sum()
        sim = pearson_on_overlap(matrix, target_user, other)
        if overlap >= min_overlap and not pd.isna(sim):
            rows.append({"neighbor": other, "similarity": sim, "overlap": int(overlap)})
    return pd.DataFrame(rows).sort_values("similarity", ascending=False)

user_neighbors(rating_matrix, "Karen")


In [ ]:
def predict_user_user(matrix, target_user, item, k=3, min_overlap=2, positive_only=True):
    target_mean = matrix.loc[target_user].mean()
    neighbors = user_neighbors(matrix, target_user, min_overlap=min_overlap)
    neighbors = neighbors[neighbors["neighbor"].map(lambda u: not pd.isna(matrix.loc[u, item]))]
    if positive_only:
        neighbors = neighbors[neighbors["similarity"] > 0]
    neighbors = neighbors.head(k)
    if neighbors.empty:
        return np.nan, neighbors

    numerator = 0.0
    denominator = 0.0
    rows = []
    for _, row in neighbors.iterrows():
        u = row["neighbor"]
        sim = row["similarity"]
        neighbor_mean = matrix.loc[u].mean()
        centered_rating = matrix.loc[u, item] - neighbor_mean
        contribution = sim * centered_rating
        numerator += contribution
        denominator += abs(sim)
        rows.append({
            "neighbor": u,
            "similarity": sim,
            "neighbor_rating": matrix.loc[u, item],
            "neighbor_mean": neighbor_mean,
            "centered_rating": centered_rating,
            "weighted_contribution": contribution,
        })
    prediction = target_mean + numerator / denominator if denominator else np.nan
    return prediction, pd.DataFrame(rows)

pred, evidence = predict_user_user(rating_matrix, "Karen", "Independence Day", k=3)
print(f"Predicted Karen rating for Independence Day: {pred:.2f}")
evidence.round(3)


In [ ]:
def recommend_user_user(matrix, target_user, n=5, k=3):
    unseen_items = matrix.columns[matrix.loc[target_user].isna()]
    rows = []
    for item in unseen_items:
        pred, evidence = predict_user_user(matrix, target_user, item, k=k)
        if not pd.isna(pred):
            rows.append({
                "user": target_user,
                "recommended_movie": item,
                "predicted_rating": pred,
                "supporting_neighbors": ", ".join(evidence["neighbor"].tolist()),
            })
    return pd.DataFrame(rows).sort_values("predicted_rating", ascending=False).head(n)

recommend_user_user(rating_matrix, "Karen", n=5, k=3).round(2)


In [ ]:
pred, evidence = predict_user_user(rating_matrix, "Alice", "Blade Runner", k=3)
print("Already rated items are normally filtered out for recommendation.")
evidence.round(3)


Exercises:
1. Predict a rating for Alice on Toy Story.
2. Compare k=1 and k=3. Which explanation is easier to trust?
